### Simplificação da rede viária 

Resultado esperado:
- Geometria simplificada com o uso da biblioteca neatnet.neatify 
- GeoPackage/GeoParquet com a nova geometria

In [25]:
# Instala dependencias
%pip install geopandas pyogrio shapely neatnet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Bibliotecas 
import geopandas as gpd
import pyogrio as ogr
import shapely as shp
import neatnet
import pathlib
import matplotlib.pyplot as plt
import osmnx as ox

import networkx as nx

%matplotlib inline


Carrega rede

In [9]:
# Rede Bom Fim
graph = ox.load_graphml("../data/graph/rede-bomFim.graphml")


Carrega origens

In [10]:
# Carrega origens
origens = gpd.read_file(
    "../data/o-d/origens_porto_alegre.gpkg",
    layer="origens"
)

In [11]:
# Verificar se a rede é formada por mais de um componente
componentes = list(nx.weakly_connected_components(graph))

print("Número de componentes:", len(componentes))


Número de componentes: 1


Se nmr de componentes > 1

In [12]:
# Gera os componentes fracamente conectados
componentes = sorted(
    nx.weakly_connected_components(graph),
    key=len,
    reverse=True
)

# Quantos nós tem cada componente
for i, componente in enumerate(componentes):
    print(f"Componente {i}: {len(componente)} nós")

Componente 0: 55 nós


-------------------------
**Exemplo de resultado**

Componente 0: 125430 nós

Componente 1: 18 nós

Componente 2: 7 nós

Componente 3: 3 nós

*Interpretação: Isso já sugere que o componente 0 é a rede principal e os outros são pequenos trechos isolados*

-------------------------

Verifica se as origens estão na rede principal

In [22]:
# Carrega arquivo com os barirros
bairros = gpd.read_file("../data/bairros/Bairros_LC12112_16.shp")

# Verifica sistema de coordenadas
print(bairros.crs)

# Converte o sistema de coordenadas para EPSG:31982
bairros = bairros.set_crs("EPSG:31982", allow_override=True)

# Converte skp para gpkg
bairros.to_file(
    "../data/bairros/bairros.gpkg",
    layer="bairros",
    driver="GPKG"
)

bairros.head()

PROJCS["TM-POA",GEOGCS["SIRGAS 2000",DATUM["Sistema_de_Referencia_Geocentrico_para_las_AmericaS_2000",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6674"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",-51],PARAMETER["scale_factor",0.999995],PARAMETER["false_easting",300000],PARAMETER["false_northing",5000000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]


,OBJECTID,CODIGO,NOME,EDITOR,DATA_EDICA,GEOM_AREA,GEOM_LEN,geometry
0,1281,0,PASSO DAS PEDRAS,None,2016-10-20,0.0,0.0,"POLYGON ((287595.213 1678609.133, 287635.971 1..."
1,1282,0,HIGIENÓPOLIS,None,2016-10-20,0.0,0.0,"POLYGON ((282819.328 1677715.723, 282809.697 1..."
2,1283,0,VILA IPIRANGA,None,2016-10-20,0.0,0.0,"POLYGON ((285882.786 1676864.358, 285806.568 1..."
3,1288,0,PARQUE SANTA FÉ,None,2016-10-20,0.0,0.0,"POLYGON ((290475.978 1679356.214, 290477.321 1..."
4,1289,0,COSTA E SILVA,None,2016-10-20,0.0,0.0,"POLYGON ((289403.688 1679659.969, 289400.081 1..."


In [ ]:
# REVER
# Seleciona o Bom Fim
bomfim = bairros[bairros["NOME"] == "BOM FIM"]
origens_bomfim = origens[origens.geometry.within(bomfim.geometry.iloc[0])]

origens_bomfim.to_file(
    "../data/o-d/origens_bomfim.gpkg",
    layer="origens_bomfim",
    driver="GPKG"
)


Número de origens no Bom Fim: 0


In [18]:

# Cria um dicionário que mapeia cada nó para o índice do componente ao qual ele pertence
node_to_component = {}

for i, componente in enumerate(componentes):
    for node in componente:
        node_to_component[node] = i

origens["componente"] = origens["node"].map(node_to_component)

print(origens[["origin_id", "node", "componente"]])
origens["componente"].unique()

      origin_id        node  componente
0         235.0  2248442166         NaN
1         236.0   296249745         NaN
2         237.0  2248442166         NaN
3         238.0   508060554         NaN
4         239.0   583532859         NaN
...         ...         ...         ...
6184    12866.0  8658562772         NaN
6185    12869.0  8658562772         NaN
6186    12871.0  8658562772         NaN
6187    12872.0  8658562772         NaN
6188    12875.0  1011917815         NaN

[6189 rows x 3 columns]


array([nan,  0.])

In [22]:
print(type(graph))
node = origens.iloc[0]["node"]

print(node)
print(node in graph.nodes)
origens["node"].isin(graph.nodes).value_counts()

<class 'networkx.classes.multidigraph.MultiDiGraph'>
2248442166
False


node
False    6174
True       15
Name: count, dtype: int64